# 2022 Dataset — Feature Validation and Data Quality

## Objective
Validate the converted 2022 flow-level dataset before machine-learning experiments.

## Dataset
- Source: USBVPN2022
- Target classes: Normal, OpenVPN, WireGuard
- Unit of analysis: One row per flow
- Target column: `label`
- Data file: `data/processed/2022/2022_three_class_flows.parquet`
- Provenance file: `results/2022_source_provenance.csv`

## Validation principles
- Preserve the original converted Parquet.
- Do not use source filenames, application groups, IP addresses, ports, or protocol identifiers as model features.
- Do not delete duplicate flows or change labels without evidence.
- Investigate missing values and class imbalance before training.
- Keep preprocessing and imputation inside the training pipeline to avoid data leakage.

## Scope
This notebook documents dataset quality and limitations. It does not train models or guarantee future model accuracy.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()

if not (ROOT / "data/processed/2022/2022_three_class_flows.parquet").exists():
    ROOT = Path.home() / "vpn-traffic-fyp"

PARQUET_PATH = ROOT / "data/processed/2022/2022_three_class_flows.parquet"
PROVENANCE_PATH = ROOT / "results/2022_source_provenance.csv"

assert PARQUET_PATH.exists(), f"Parquet not found: {PARQUET_PATH}"
assert PROVENANCE_PATH.exists(), f"Provenance not found: {PROVENANCE_PATH}"

df = pd.read_parquet(PARQUET_PATH)
provenance = pd.read_csv(PROVENANCE_PATH)

print("Project directory:", ROOT)
print("Dataset shape:", df.shape)
print("Provenance shape:", provenance.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClass counts:")
print(df["label"].value_counts())

print("\nLabel alignment with provenance:",
      df["label"].reset_index(drop=True).equals(
          provenance["label"].reset_index(drop=True)
      ))


Project directory: /home/alihussain/vpn-traffic-fyp
Dataset shape: (53069, 14)
Provenance shape: (53069, 4)

Columns:
['label', 'packet_record_count', 'forward_bytes', 'backward_bytes', 'signed_bytes_sum', 'total_bytes', 'packet_size_mean', 'packet_size_std', 'packet_size_min', 'packet_size_max', 'duration_s', 'iat_mean_s', 'iat_std_s', 'iat_max_s']

Class counts:
label
Normal       50191
WireGuard     1758
OpenVPN       1120
Name: count, dtype: int64

Label alignment with provenance: True


In [2]:
# Feature-level missingness
features = df.drop(columns=["label"])

missing_report = pd.DataFrame({
    "missing_count": features.isna().sum(),
    "missing_percent": (features.isna().mean() * 100).round(4)
})

print("===== MISSING VALUES =====")
display(missing_report)

# Infinite numeric values
numeric_features = features.select_dtypes(include="number")
infinite_counts = pd.Series(
    np.isinf(numeric_features.to_numpy()).sum(axis=0),
    index=numeric_features.columns,
    name="infinite_count"
)

print("\n===== INFINITE VALUES =====")
display(infinite_counts.to_frame())

# Confirm whether all three IAT fields are missing on the same rows
iat_cols = ["iat_mean_s", "iat_std_s", "iat_max_s"]

print(
    "\nAll IAT fields missing on the same rows:",
    df[iat_cols].isna().nunique(axis=1).eq(1).all()
)

print(
    "Rows with missing IAT:",
    int(df["iat_mean_s"].isna().sum())
)

print("\n===== PACKET COUNT FOR MISSING IAT =====")
display(
    df.loc[df["iat_mean_s"].isna(), "packet_record_count"]
      .value_counts()
      .sort_index()
      .rename_axis("packet_record_count")
      .to_frame("flow_count")
)


===== MISSING VALUES =====


,missing_count,missing_percent
packet_record_count,0,0.0000
forward_bytes,0,0.0000
backward_bytes,0,0.0000
signed_bytes_sum,0,0.0000
total_bytes,0,0.0000
packet_size_mean,0,0.0000
packet_size_std,0,0.0000
packet_size_min,0,0.0000
packet_size_max,0,0.0000
duration_s,0,0.0000



===== INFINITE VALUES =====


,infinite_count
packet_record_count,0
forward_bytes,0
backward_bytes,0
signed_bytes_sum,0
total_bytes,0
packet_size_mean,0
packet_size_std,0
packet_size_min,0
packet_size_max,0
duration_s,0



All IAT fields missing on the same rows: True
Rows with missing IAT: 2895

===== PACKET COUNT FOR MISSING IAT =====


,flow_count
packet_record_count,
1,2895


In [3]:

# 1. Class distribution
class_report = (
    df["label"]
    .value_counts()
    .rename_axis("label")
    .to_frame("flow_count")
)

class_report["percentage"] = (
    class_report["flow_count"] / len(df) * 100
).round(4)

print("===== CLASS BALANCE =====")
display(class_report)


# 2. Exact duplicate rows, including the target label
duplicate_count = int(df.duplicated().sum())

print(
    "\nExact duplicate rows (excluding first occurrence):",
    duplicate_count
)


# 3. Feature patterns associated with multiple labels
feature_cols = [c for c in df.columns if c != "label"]

label_counts_per_pattern = (
    df.groupby(feature_cols, dropna=False)["label"]
      .nunique()
)

conflicting_patterns = label_counts_per_pattern[
    label_counts_per_pattern > 1
]

print(
    "\nFeature patterns associated with multiple labels:",
    len(conflicting_patterns)
)

# Count every row belonging to a conflicting pattern.
# This avoids the previous MultiIndex .loc KeyError.
row_label_counts = df.groupby(
    feature_cols,
    dropna=False
)["label"].transform("nunique")

conflict_rows = df.loc[row_label_counts > 1].copy()

print(
    "Rows belonging to conflicting patterns:",
    len(conflict_rows)
)

if not conflict_rows.empty:
    print("\nConflicting rows by label:")
    display(
        conflict_rows.groupby("label")
        .size()
        .to_frame("flow_count")
    )


# 4. Mathematical consistency checks
checks = {
    "total_bytes_equals_forward_plus_backward": np.allclose(
        df["total_bytes"],
        df["forward_bytes"] + df["backward_bytes"],
        equal_nan=True
    ),
    "signed_sum_equals_forward_minus_backward": np.allclose(
        df["signed_bytes_sum"],
        df["forward_bytes"] - df["backward_bytes"],
        equal_nan=True
    ),
    "mean_equals_total_over_record_count": np.allclose(
        df["packet_size_mean"],
        df["total_bytes"] / df["packet_record_count"],
        equal_nan=True
    )
}

print("\n===== MATHEMATICAL CONSISTENCY =====")

for name, passed in checks.items():
    print(f"{name}: {passed}")


===== CLASS BALANCE =====


,flow_count,percentage
label,,
Normal,50191,94.5769
WireGuard,1758,3.3127
OpenVPN,1120,2.1105



Exact duplicate rows (excluding first occurrence): 5303

Feature patterns associated with multiple labels: 2
Rows belonging to conflicting patterns: 49

Conflicting rows by label:


,flow_count
label,
Normal,6
WireGuard,43



===== MATHEMATICAL CONSISTENCY =====
total_bytes_equals_forward_plus_backward: True
signed_sum_equals_forward_minus_backward: True
mean_equals_total_over_record_count: True


## Data Quality Findings and Decisions

### 1. Dataset integrity
- The processed dataset contains 53,069 flows and 14 columns: one target label and 13 numerical features.
- The provenance manifest contains 53,069 rows, and its labels align row-by-row with the Parquet dataset.
- The dataset contains three target classes: Normal, OpenVPN, and WireGuard.
- These checks establish structural consistency; they do not independently prove that every original source label is correct.

### 2. Missing inter-arrival-time (IAT) values
- The three IAT features each contain 2,895 missing values.
- All three features are missing on exactly the same rows.
- All affected rows have a packet record count of one.
- Decision: preserve these flows. Handle missing values inside the model's training pipeline after splitting the data.

### 3. Class imbalance
- Normal: 50,191 flows (94.5769%).
- WireGuard: 1,758 flows (3.3127%).
- OpenVPN: 1,120 flows (2.1105%).
- Decision: report macro-F1, balanced accuracy, per-class recall, and confusion matrices in addition to accuracy.

### 4. Duplicate and conflicting patterns
- There are 5,303 duplicate rows when duplicates are counted excluding the first occurrence and including the target label.
- Two feature patterns are associated with multiple target labels, covering 49 rows.
- One pattern represents 47 single-record flows; the other represents two flows with 60 packet records each.
- Decision: do not automatically remove duplicates or change labels. Investigate their effect on evaluation and document any limitations.

### 5. Mathematical consistency
The following checks passed:
- total_bytes = forward_bytes + backward_bytes
- signed_bytes_sum = forward_bytes - backward_bytes
- packet_size_mean = total_bytes / packet_record_count

These relationships show that some features are mathematically redundant. Retain the original processed dataset unchanged and explicitly define the feature set used in each experiment.

### 6. Leakage prevention
- Source filenames, application groups, row indices, IP addresses, ports, and protocol identifiers must not be model input features.
- Provenance metadata may be used for auditing and carefully designed evaluation splits only.
- Fit imputation and any other learned preprocessing on training data only.

### 7. Limitations and readiness
This dataset has passed the structural and feature-consistency checks documented above. However, source-label correctness, duplicate-related split effects, and generalization to unseen applications or later capture periods still require careful evaluation.

The dataset is not evidence of guaranteed model accuracy. Model evaluation must use a documented split strategy and class-sensitive metrics.


In [4]:
# Review mathematically redundant features before selecting model inputs

print("===== FEATURE RELATIONSHIPS =====")

relationships = {
    "total_bytes = forward_bytes + backward_bytes":
        np.allclose(
            df["total_bytes"],
            df["forward_bytes"] + df["backward_bytes"]
        ),
    "signed_bytes_sum = forward_bytes - backward_bytes":
        np.allclose(
            df["signed_bytes_sum"],
            df["forward_bytes"] - df["backward_bytes"]
        ),
    "packet_size_mean = total_bytes / packet_record_count":
        np.allclose(
            df["packet_size_mean"],
            df["total_bytes"] / df["packet_record_count"]
        )
}

for relationship, passed in relationships.items():
    print(f"{relationship}: {passed}")

print("\n===== CURRENT FEATURE LIST =====")
for i, col in enumerate([c for c in df.columns if c != "label"], start=1):
    print(f"{i:02d}. {col}")


===== FEATURE RELATIONSHIPS =====
total_bytes = forward_bytes + backward_bytes: True
signed_bytes_sum = forward_bytes - backward_bytes: True
packet_size_mean = total_bytes / packet_record_count: True

===== CURRENT FEATURE LIST =====
01. packet_record_count
02. forward_bytes
03. backward_bytes
04. signed_bytes_sum
05. total_bytes
06. packet_size_mean
07. packet_size_std
08. packet_size_min
09. packet_size_max
10. duration_s
11. iat_mean_s
12. iat_std_s
13. iat_max_s


### Model Feature Selection

The original 2022 Parquet dataset is preserved unchanged. Three mathematically derived features—`signed_bytes_sum`, `total_bytes`, and `packet_size_mean`—are excluded from the initial model input because they can be calculated exactly from other retained features.

The initial model feature set contains 10 features: `packet_record_count`, `forward_bytes`, `backward_bytes`, `packet_size_std`, `packet_size_min`, `packet_size_max`, `duration_s`, `iat_mean_s`, `iat_std_s`, and `iat_max_s`.

This selection reduces redundant inputs, particularly for Logistic Regression. It does not guarantee improved performance; feature selection will be assessed during model evaluation. Missing IAT values will be handled within the training pipeline rather than by modifying the original dataset.

In [5]:
print("===== FINAL MODEL FEATURE AUDIT =====")

excluded_features = [
    "signed_bytes_sum",
    "total_bytes",
    "packet_size_mean",
]

model_features = [
    col for col in df.columns
    if col != "label" and col not in excluded_features
]

print("Total original features:", len(df.columns) - 1)
print("Excluded derived features:", excluded_features)
print("Selected model features:", len(model_features))

for i, col in enumerate(model_features, start=1):
    print(f"{i:02d}. {col}")

print("\n===== VALIDATION =====")
print("All selected features numeric:",
      all(pd.api.types.is_numeric_dtype(df[col]) for col in model_features))
print("Label excluded from model features:", "label" not in model_features)
print("Original dataset columns preserved:", len(df.columns) == 14)
print("Missing values retained in original dataset:",
      int(df[model_features].isna().sum().sum()))

===== FINAL MODEL FEATURE AUDIT =====
Total original features: 13
Excluded derived features: ['signed_bytes_sum', 'total_bytes', 'packet_size_mean']
Selected model features: 10
01. packet_record_count
02. forward_bytes
03. backward_bytes
04. packet_size_std
05. packet_size_min
06. packet_size_max
07. duration_s
08. iat_mean_s
09. iat_std_s
10. iat_max_s

===== VALIDATION =====
All selected features numeric: True
Label excluded from model features: True
Original dataset columns preserved: True
Missing values retained in original dataset: 8685
